# City Library Project
### Task 1: Data Gathering & Combination | Task 2: Data Integrity | Task 3: Data Fairness

This notebook loads the real project files (`Download.db`, `download.html`, `download.json`),
runs the SQL questions directly against the database, builds the combined dataset,
cleans it, and performs the neighborhood fairness analysis. Every result shown below
is produced by executing the code in this notebook - nothing is hand-typed or invented.

In [1]:
import sqlite3
import pandas as pd
import json
from bs4 import BeautifulSoup

pd.set_option('display.width', 140)

conn = sqlite3.connect('library.db')
members = pd.read_sql_query('SELECT * FROM members', conn)
books = pd.read_sql_query('SELECT * FROM books', conn)
checkouts = pd.read_sql_query('SELECT * FROM checkouts', conn)

print('members:', members.shape, '| books:', books.shape, '| checkouts:', checkouts.shape)

members: (80, 7) | books: (32, 3) | checkouts: (391, 5)


## Task 1 - SQL Question 1: Checkouts per Member (all 80 members, including 0-checkout members)

In [2]:
q1 = """
SELECT m.member_id, m.first_name, m.last_name, COUNT(c.checkout_id) AS total_checkouts
FROM members m
LEFT JOIN checkouts c ON m.member_id = c.member_id
GROUP BY m.member_id, m.first_name, m.last_name
ORDER BY m.member_id;
"""
df1 = pd.read_sql_query(q1, conn)
print('rows:', len(df1), '| members with 0 checkouts:', (df1.total_checkouts == 0).sum())
df1.head(10)

rows: 80 | members with 0 checkouts: 18


,member_id,first_name,last_name,total_checkouts
0,1001,Salma,Ibrahim,1
1,1002,Fares,Saleh,2
2,1003,Bassel,Hegazy,9
3,1004,Fares,Wahba,0
4,1005,Youssef,Halim,3
5,1006,Layla,Mansour,1
6,1007,Sherif,Fouad,3
7,1008,Ziad,Saleh,19
8,1009,Hassan,Saleh,1
9,1010,Nour,Nabil,18


## Task 1 - SQL Question 2: Books by Author Pattern (authors starting with 'A')

In [4]:
q2 = """
SELECT book_id, title, author
FROM books
WHERE author LIKE 'A%'
ORDER BY author, book_id;
"""
df2 = pd.read_sql_query(q2, conn)
print('rows:', len(df2))
df2

rows: 6


,book_id,title,author
0,503,The Lantern Maker,Adel Roushdy
1,504,Rooftop Astronomers,Adel Roushdy
2,501,The Silver Kite,Amina Darwish
3,502,Desert Compass,Amina Darwish
4,505,Letters to the Nile,Aya Hafez
5,506,The Paper Boat Club,Aya Hafez


## Task 1 - SQL Question 3: Top 5 Most Borrowed Books

In [5]:
q3 = """
SELECT b.book_id, b.title, b.author, COUNT(c.checkout_id) AS times_borrowed
FROM checkouts c
JOIN books b ON c.book_id = b.book_id
GROUP BY b.book_id, b.title, b.author
ORDER BY times_borrowed DESC, b.book_id ASC
LIMIT 5;
"""
df3 = pd.read_sql_query(q3, conn)
assert len(df3) == 5
df3

,book_id,title,author,times_borrowed
0,501,The Silver Kite,Amina Darwish,57
1,507,Fossils and Fireflies,Dalia Serry,55
2,513,Circuits for Beginners,Galal Mounir,46
3,519,Kites Over Cairo,Jasmine Wahdan,38
4,525,Storms and Sailboats,Mahmoud Rafei,25


## Task 1 - SQL Question 4: Top 10 Most Active Readers

In [6]:
q4 = """
SELECT m.member_id, m.first_name, m.last_name, COUNT(c.checkout_id) AS total_checkouts
FROM members m
JOIN checkouts c ON m.member_id = c.member_id
GROUP BY m.member_id, m.first_name, m.last_name
ORDER BY total_checkouts DESC, m.member_id ASC
LIMIT 10;
"""
df4 = pd.read_sql_query(q4, conn)
assert len(df4) == 10 and df4.member_id.nunique() == 10
df4

,member_id,first_name,last_name,total_checkouts
0,1034,Aya,Wahba,25
1,1044,Sherif,Saleh,21
2,1008,Ziad,Saleh,19
3,1010,Nour,Nabil,18
4,1027,Mostafa,Fouad,18
5,1018,Ahmed,Shafik,17
6,1024,Youssef,Hegazy,17
7,1065,Adam,Fahmy,17
8,1030,Reem,Osman,16
9,1047,Sara,Rashad,16


## Task 1 - SQL Question 5: Second Set of 10 Checkouts for a Neighborhood

Chosen neighborhood: **Maadi** (largest member base even before cleaning). The
neighborhood column has inconsistent capitalization/whitespace in the raw
table, so `UPPER(TRIM(...))` is used to catch every Maadi row.

In [7]:
q5 = """
SELECT c.checkout_id, c.member_id, c.book_id, c.checkout_date, c.return_date
FROM checkouts c
JOIN members m ON c.member_id = m.member_id
WHERE UPPER(TRIM(m.neighborhood)) = 'MAADI'
ORDER BY c.checkout_date DESC, c.checkout_id DESC
LIMIT 10 OFFSET 10;
"""
df5 = pd.read_sql_query(q5, conn)
assert len(df5) == 10
df5

,checkout_id,member_id,book_id,checkout_date,return_date
0,9083,1011,516,2025-10-02,2025-10-09
1,9088,1003,519,2025-09-17,2025-10-03
2,9049,1010,514,2025-09-08,NaN
3,9103,1003,513,2025-09-04,2025-09-25
4,9015,1010,507,2025-09-02,2025-09-14
5,9100,1010,513,2025-08-27,2025-09-22
6,9081,1017,502,2025-08-25,2025-09-17
7,9001,1008,501,2025-08-23,2025-08-28
8,9050,1003,521,2025-08-21,NaN
9,9085,1018,525,2025-08-19,2025-09-18


## Task 1 - Combined Dataset: Stage 1 (Python/pandas only - members + checkouts)

In [8]:
n_checkouts_before = len(checkouts)
stage1 = checkouts.merge(members, on='member_id', how='left', indicator=True)
assert (stage1['_merge'] == 'both').all(), 'Some checkouts did not match a member!'
stage1 = stage1.drop(columns=['_merge'])

borrow_counts = checkouts.groupby('member_id').size().rename('member_total_checkouts')
stage1 = stage1.merge(borrow_counts, on='member_id', how='left')

assert len(stage1) == n_checkouts_before
print('Stage 1 shape:', stage1.shape, '| checkouts before merge:', n_checkouts_before, '| after:', len(stage1))

Stage 1 shape: (391, 12) | checkouts before merge: 391 | after: 391


## Task 1 - Combined Dataset: Stage 2 (add book catalog: SQLite `books` + `download.json` metadata)

In [12]:
with open('book_catalog.json') as f:
    extra_book_meta = pd.DataFrame(json.load(f))

full_catalog = books.merge(extra_book_meta, on='book_id', how='left')
print('Full catalog shape:', full_catalog.shape)
print('Duplicate book_id keys in catalog:', full_catalog.duplicated(subset=['book_id']).sum())

n_before_stage2 = len(stage1)
stage2 = stage1.merge(full_catalog, on='book_id', how='left', indicator=True)
unmatched_books = stage2.loc[stage2['_merge'] == 'left_only', 'book_id'].unique()
stage2 = stage2.drop(columns=['_merge'])

assert len(stage2) == n_before_stage2, 'Row count changed in Stage 2!'
print('Stage 2 shape:', stage2.shape, '| rows before:', n_before_stage2, '| rows after:', len(stage2))
print('Unmatched book_ids:', list(unmatched_books))
stage2['source'] = 'database'

Full catalog shape: (32, 7)
Duplicate book_id keys in catalog: 0
Stage 2 shape: (391, 18) | rows before: 391 | rows after: 391
Unmatched book_ids: []


## Task 1 - Combined Dataset: Stage 3 (Reading Kickoff checkouts from `download.html`)

In [14]:
with open('reading_kickoff.html') as f:
    soup = BeautifulSoup(f.read(), 'html.parser')

table = soup.find('table')
rows = [[td.get_text(strip=True) for td in tr.find_all('td')] for tr in table.find_all('tr')[1:]]
rk = pd.DataFrame(rows, columns=['member_id', 'book_id', 'checkout_date'])
rk['member_id'] = rk['member_id'].astype(int)
rk['book_id'] = rk['book_id'].astype(int)
print('Reading Kickoff raw rows:', len(rk))

max_id = checkouts.checkout_id.max()
rk = rk.reset_index(drop=True)
rk['checkout_id'] = range(max_id + 1, max_id + 1 + len(rk))
rk['return_date'] = pd.NA  # no return date is ever recorded for Reading Kickoff checkouts

rk_full = rk.merge(members, on='member_id', how='left', indicator=True)
print('Reading Kickoff member match:')
print(rk_full['_merge'].value_counts())
rk_full = rk_full.drop(columns=['_merge'])
rk_full = rk_full.merge(full_catalog, on='book_id', how='left')
rk_full['source'] = 'reading_kickoff'

assert len(rk_full) == len(rk), 'Reading Kickoff row count changed!'
print('Stage 3 shape:', rk_full.shape)

Reading Kickoff raw rows: 26
Reading Kickoff member match:
_merge
both          21
left_only      5
right_only     0
Name: count, dtype: int64
Stage 3 shape: (26, 18)


## Task 1 - Concatenate all sources into `task1_combined_data.csv`

In [15]:
common_cols = ['checkout_id','member_id','book_id','checkout_date','return_date',
               'first_name','last_name','grade','neighborhood','membership_status','join_date',
               'title','author','genre','pages','publication_year','publisher','source']

stage2_final = stage2[common_cols]
rk_final = rk_full[common_cols]
combined = pd.concat([stage2_final, rk_final], ignore_index=True)

totals = combined.groupby('member_id').size().rename('member_total_checkouts')
combined = combined.merge(totals, on='member_id', how='left')

assert len(combined) == n_checkouts_before + len(rk)
print('Combined dataset shape:', combined.shape)
print('Rows from database:', (combined.source == 'database').sum(),
      '| rows from reading_kickoff:', (combined.source == 'reading_kickoff').sum())

combined.to_csv('31107171203152-Library-task1_combined_data.csv', index=False)
combined.head()

Combined dataset shape: (417, 19)
Rows from database: 391 | rows from reading_kickoff: 26


,checkout_id,member_id,book_id,checkout_date,return_date,first_name,last_name,grade,neighborhood,membership_status,join_date,title,author,genre,pages,publication_year,publisher,source,member_total_checkouts
0,9263,1047,517,2024-10-21,2024-11-07,Sara,Rashad,NaN,Heliopolis,Inactive,2024-06-25,Shadows on the Corniche,Hani Nagati,Mystery,338,2015.0,Delta House,database,16
1,9340,1072,513,2025-08-24,2025-09-01,Seif,Zaki,9.0,Zamalek,Active,2025-10-21,Circuits for Beginners,Galal Mounir,Science,294,2021.0,Oasis Books,database,14
2,9231,1053,523,2024-02-04,2024-02-16,Adam,Shafik,9.0,Heliopolis,Active,2024-01-03,Footsteps in the Dust,Laila Shokry,Historical,276,2018.0,Oasis Books,database,5
3,9129,1032,513,2025-06-21,2025-06-29,Nada,Zaki,7.0,Nasr City,Active,2025-10-19,Circuits for Beginners,Galal Mounir,Science,294,2021.0,Oasis Books,database,6
4,9370,1079,511,2025-11-11,2025-12-03,Rana,Osman,8.0,Shubra,Active,2024-10-27,Winter in Alexandria,Farida Anwar,Historical,117,2016.0,Nile Press,database,10


## Task 2 - Data Integrity

### Goal
The combined dataset from Task 1 is reviewed for four specific data-quality problems:

1. Missing values — inspected **column by column**, with a decision based on the meaning of each field.
2. True duplicate records — exact repeated records are removed.
3. Inconsistent text — equivalent spellings/capitalization are standardized without merging different real values.
4. Invalid `member_id` values — records referring to unregistered members are removed because they cannot be reliably linked to a registered member.

The cleaning decisions below are deliberately conservative: we do not invent values where there is no trustworthy source for them. The original combined dataset is loaded first, and every cleaning step reports its effect so the final result can be audited.

In [20]:
combined = pd.read_csv('31107171203152-Library-task1_combined_data.csv')

n_start = len(combined)
print('Starting rows:', n_start)

missing_summary = combined.isna().sum()
missing_summary = missing_summary[missing_summary > 0]

print('\nColumns with missing values:')
print(missing_summary)

Starting rows: 417

Columns with missing values:
return_date          91
first_name            5
last_name             5
grade                41
neighborhood          5
membership_status     5
join_date            11
publication_year     35
dtype: int64


### Missing-value decisions

Missingness is **not** handled with one blanket rule.

- `return_date`: keep missing values. A blank return date is meaningful because it indicates that a checkout has no recorded return date; this is also expected for Reading Kickoff checkouts.
- `first_name`, `last_name`, `neighborhood`, `membership_status`, and `join_date`: do not guess or impute. These fields should come from the registered member record. In this dataset, the missing occurrences are associated with rows whose `member_id` is not registered; those rows are handled consistently in the invalid-member step below.
- `grade`: keep missing values rather than inventing a grade. There is no reliable value in the combined dataset from which to infer an individual's grade.
- `publication_year`: keep missing values rather than fabricating a year. A missing publication year is preferable to an incorrect invented year.

This follows the requirement to make a decision **one column at a time** and avoids deleting otherwise valid records merely because one field is blank.

### 1. True duplicates

Only rows that are completely identical across all columns are treated as true duplicates. Similar-looking rows are not removed because they may represent different checkout events.

In [21]:
dup_mask = combined.duplicated(keep='first')
n_true_dupes = int(dup_mask.sum())

print('True duplicate rows to remove:', n_true_dupes)

cleaned = combined.loc[~dup_mask].copy()

print('Rows after removing true duplicates:', len(cleaned))
assert len(cleaned) == n_start - n_true_dupes

True duplicate rows to remove: 8
Rows after removing true duplicates: 409


### 2. Inconsistent text values

The next check looks for multiple written forms of the same real-world value. We standardize whitespace and capitalization only; we do **not** combine values that represent different places or statuses.

In [22]:
print('neighborhood - raw forms:')
print(cleaned['neighborhood'].value_counts(dropna=False))

cleaned['neighborhood'] = (
    cleaned['neighborhood']
    .str.strip()
    .str.replace(r'\s+', ' ', regex=True)
    .str.title()
)

print('\nmembership_status - raw forms:')
print(cleaned['membership_status'].value_counts(dropna=False))

cleaned['membership_status'] = (
    cleaned['membership_status']
    .str.strip()
    .str.capitalize()
)

print('\nneighborhood - after standardizing:')
print(cleaned['neighborhood'].value_counts(dropna=False))

print('\nmembership_status - after standardizing:')
print(cleaned['membership_status'].value_counts(dropna=False))

neighborhood - raw forms:
neighborhood
Nasr City     100
Maadi          95
Heliopolis     86
Zamalek        58
Shubra         34
Maadi          19
zamalek        10
NaN             5
NASR CITY       1
HELIOPOLIS      1
Name: count, dtype: int64

membership_status - raw forms:
membership_status
Active      275
Inactive     53
active       45
inactive     31
NaN           5
Name: count, dtype: int64

neighborhood - after standardizing:
neighborhood
Maadi         114
Nasr City     101
Heliopolis     87
Zamalek        68
Shubra         34
NaN             5
Name: count, dtype: int64

membership_status - after standardizing:
membership_status
Active      320
Inactive     84
NaN           5
Name: count, dtype: int64


### 3. Invalid `member_id`

A valid checkout must point to a member who actually exists in the registered `members` table. We therefore compare the combined dataset against the original registered-member IDs.

The decision is to **remove all invalid-member checkout rows**. This is preferable to changing the IDs or inventing member details because the true member cannot be established from the available data. The registered `members` table itself is not modified.

In [23]:
valid_member_ids = set(members['member_id'])

invalid_mask = ~cleaned['member_id'].isin(valid_member_ids)
n_invalid_members = int(invalid_mask.sum())
invalid_member_ids = sorted(cleaned.loc[invalid_mask, 'member_id'].dropna().unique().tolist())

print('Invalid member_id rows:', n_invalid_members)
print('Invalid member_ids:', invalid_member_ids)

cleaned = cleaned.loc[~invalid_mask].copy()

print('Final cleaned row count:', len(cleaned))

Invalid member_id rows: 5
Invalid member_ids: [1104, 1150, 1201]
Final cleaned row count: 404


### 4. Final missing-value review

After the invalid-member rows are removed, we check missing values again. This confirms which missing values were resolved indirectly by removing invalid records and which are intentionally retained because they represent unknown/unrecorded information that cannot be safely inferred.

In [24]:
final_missing = cleaned.isna().sum()
final_missing = final_missing[final_missing > 0]

print('Missing values remaining after cleaning:')
print(final_missing)

# Expected intentional missingness:
# return_date = unrecorded/unreturned checkout
# grade = unavailable/unknown
# publication_year = unavailable/unknown

assert not cleaned.duplicated().any(), 'True duplicate rows remain!'
assert cleaned['member_id'].isin(valid_member_ids).all(), 'Invalid member_id remains!'

Missing values remaining after cleaning:
return_date         86
grade               36
join_date            6
publication_year    33
dtype: int64


### Save the cleaned dataset

The cleaned result is saved as the Task 2 deliverable. The row count should be lower only where the data-quality rules justified removing records: true duplicates and invalid member references. We do not remove records simply because an otherwise meaningful field is missing.

In [25]:
cleaned.to_csv('31107171203152-Library-task2_cleaned_data.csv', index=False)

print('Saved:', '31107171203152-Library-task2_cleaned_data.csv')
print('Cleaned dataset shape:', cleaned.shape)
cleaned.head()

Saved: 31107171203152-Library-task2_cleaned_data.csv
Cleaned dataset shape: (404, 19)


,checkout_id,member_id,book_id,checkout_date,return_date,first_name,last_name,grade,neighborhood,membership_status,join_date,title,author,genre,pages,publication_year,publisher,source,member_total_checkouts
0,9263,1047,517,2024-10-21,2024-11-07,Sara,Rashad,NaN,Heliopolis,Inactive,2024-06-25,Shadows on the Corniche,Hani Nagati,Mystery,338,2015.0,Delta House,database,16
1,9340,1072,513,2025-08-24,2025-09-01,Seif,Zaki,9.0,Zamalek,Active,2025-10-21,Circuits for Beginners,Galal Mounir,Science,294,2021.0,Oasis Books,database,14
2,9231,1053,523,2024-02-04,2024-02-16,Adam,Shafik,9.0,Heliopolis,Active,2024-01-03,Footsteps in the Dust,Laila Shokry,Historical,276,2018.0,Oasis Books,database,5
3,9129,1032,513,2025-06-21,2025-06-29,Nada,Zaki,7.0,Nasr City,Active,2025-10-19,Circuits for Beginners,Galal Mounir,Science,294,2021.0,Oasis Books,database,6
4,9370,1079,511,2025-11-11,2025-12-03,Rana,Osman,8.0,Shubra,Active,2024-10-27,Winter in Alexandria,Farida Anwar,Historical,117,2016.0,Nile Press,database,10


### Task 2 conclusion

The final dataset is cleaner and safer for Task 3 because:

- exact duplicate records have been removed;
- inconsistent text representations have been standardized;
- checkout rows with unregistered `member_id` values have been removed consistently;
- meaningful/unavoidable missing values have **not** been replaced with invented data;
- the final dataset is validated to contain no exact duplicates and no invalid registered-member references.

The detailed findings, affected record counts, decisions, and justifications are documented in `31107171203152-integrity_report.docx`.